# Airport Connectivity Analysis

This notebook explores a complex network of airports in the US and Canada based on reachability.

## Network description:

This is a transportation reachability network for cities in the United States and Canada. Edges are weighted so that there is an edge from city i to city j if the estimated airline travel time is less than a threshold. The travel time includes estimated stopover delays. Due to headwinds, the network is asymmetric. We include the city metropolitan populations, latitude, and longitude in this dataset.

Link: https://snap.stanford.edu/data/reachability.html

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

%matplotlib inline

## Read Data
Read the nodes (airports) and edges (reachability connections) from the provided files.

In [2]:
print("Loading data...")
nodes_df = pd.read_csv('data/reachability-meta.csv/reachability-meta.csv')

with open('data/reachability.txt/reachability.txt', 'r') as f:
    edges_lines = [line.strip().split() for line in f if not line.startswith('#')]
    edges_df = pd.DataFrame(edges_lines, columns=['FromNodeId', 'ToNodeId', 'Weight'])
    edges_df = edges_df.astype({'FromNodeId': int, 'ToNodeId': int, 'Weight': float})

nodes_df.head()

Loading data...


,node_id,name,metro_pop,latitude,longitude
0,0,"Abbotsford, BC",133497.0,49.051575,-122.328849
1,1,"Aberdeen, SD",40878.0,45.459090,-98.487324
2,2,"Abilene, TX",166416.0,32.449175,-99.741424
3,3,"Akron/Canton, OH",701456.0,40.797810,-81.371567
4,4,"Alamosa, CO",9433.0,37.468180,-105.873599


## Create Directed Graph
Create a directed graph with airports as nodes and connections as edges. We'll also extract some key characteristics.

In [3]:
G = nx.DiGraph()

# Add nodes with attributes (lat, lon)
for _, row in nodes_df.iterrows():
    G.add_node(row['node_id'], 
               name=row['name'], 
               pop=row['metro_pop'], 
               pos=(row['longitude'], row['latitude']))

# Add edges
for _, row in edges_df.iterrows():
    G.add_edge(row['FromNodeId'], row['ToNodeId'], weight=row['Weight'])

# Plot characteristics
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

print(f"Directed Graph Characteristics:")
print(f"Number of nodes: {num_nodes}")
print(f"Number of edges: {num_edges}")

Directed Graph Characteristics:
Number of nodes: 456
Number of edges: 71959


## Plot Map with All Airports and Connections
We will define a plotting function since we are going to plot multiple geographical graphs.

In [4]:
import random
import plotly.graph_objects as go
import numpy as np

pos = nx.get_node_attributes(G, 'pos')

def plot_map(graph, title, node_color='blue', edge_color='gray', edge_alpha=0.4, max_edges=3000, directions=False, direction_start_color=None, direction_end_color=None):
    fig = go.Figure()
    
    edges = list(graph.edges())
    print(f"Total edges: {len(edges)}")
    if len(edges) > max_edges:
        edges = random.sample(edges, max_edges)
        print(f"Sampling {max_edges} edges out of {total_edges} for faster plotting." if 'total_edges' in locals() else f"Sampling {max_edges} edges out of {len(graph.edges())} for faster plotting.")
        
    if direction_start_color and direction_end_color and directions:
        edge_lons_s, edge_lats_s = [], []
        edge_lons_e, edge_lats_e = [], []
        for u, v in edges:
            x1, y1 = pos[u]
            x2, y2 = pos[v]
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2
            edge_lons_s.extend([x1, mx, None])
            edge_lats_s.extend([y1, my, None])
            edge_lons_e.extend([mx, x2, None])
            edge_lats_e.extend([my, y2, None])
            
        fig.add_trace(go.Scattergeo(
            lon=edge_lons_s, lat=edge_lats_s,
            mode='lines', line=dict(width=1, color=direction_start_color),
            opacity=edge_alpha, hoverinfo='none', showlegend=False
        ))
        fig.add_trace(go.Scattergeo(
            lon=edge_lons_e, lat=edge_lats_e,
            mode='lines', line=dict(width=1, color=direction_end_color),
            opacity=edge_alpha, hoverinfo='none', showlegend=False
        ))
        
    else:
        edge_lons, edge_lats = [], []
        for u, v in edges:
            x1, y1 = pos[u]
            x2, y2 = pos[v]
            edge_lons.extend([x1, x2, None])
            edge_lats.extend([y1, y2, None])
            
        fig.add_trace(go.Scattergeo(
            lon=edge_lons, lat=edge_lats,
            mode='lines', line=dict(width=1, color=edge_color),
            opacity=edge_alpha, hoverinfo='none', showlegend=False
        ))
        
    # Nodes
    node_lons = [pos[n][0] for n in graph.nodes()]
    node_lats = [pos[n][1] for n in graph.nodes()]
    
    hover_texts = []
    for n in graph.nodes():
        attr = graph.nodes[n]
        name = attr.get('name', str(n))
        if graph.is_directed():
            deg_str = f"In-Degree: {graph.in_degree(n)}<br>Out-Degree: {graph.out_degree(n)}"
        else:
            deg_str = f"Degree: {graph.degree(n)}"
        hover_texts.append(f"Airport: {name}<br>{deg_str}")
        
    marker_dict = dict(size=5, opacity=0.8)
    # Check if node_color is an array-like for dynamic scales
    if isinstance(node_color, (list, np.ndarray, tuple)): 
        marker_dict['color'] = node_color
        marker_dict['colorscale'] = 'RdYlGn_r'  # Reversed: low/neg (green -> out > in) to high/pos (red -> in > out)
        marker_dict['showscale'] = True
        marker_dict['colorbar'] = dict(thickness=15)
    else:
        marker_dict['color'] = node_color

    fig.add_trace(go.Scattergeo(
        lon=node_lons, lat=node_lats,
        mode='markers', text=hover_texts, hoverinfo='text',
        marker=marker_dict, showlegend=False
    ))
    
    # Calculate bounds natively
    x_coords = [p[0] for p in pos.values()]
    y_coords = [p[1] for p in pos.values()]

    fig.update_layout(
        title=title,
        geo=dict(
            projection_type='mercator',
            showland=True, landcolor='#e0f3d9',
            showocean=True, oceancolor='#aadaff',
            showcountries=True, countrycolor='#999999',
            lataxis_range=[min(y_coords)-2, max(y_coords)+2], 
            lonaxis_range=[min(x_coords)-2, max(x_coords)+2]
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        height=600
    )
    fig.show()

plot_map(G, f"All Airports and Connections\nNodes: {num_nodes}, Edges: {num_edges}", max_edges=200)

ModuleNotFoundError: No module named 'plotly'

In [ ]:


strongly_connected_components = list(nx.strongly_connected_components(G))
num_scc = len(strongly_connected_components)
largest_scc_size = max(len(c) for c in strongly_connected_components)

print("--- Directed Graph ---")
print(f"Strongly connected components: {num_scc}")
print(f"Largest strongly connected component size: {largest_scc_size}")

--- Directed Graph ---
Strongly connected components: 1
Largest strongly connected component size: 456


## Plot Map with Only One-Way Connections
Filter for connections that go in only one direction and plot them.

In [ ]:
one_way_edges = []
for u, v in G.edges():
    if not G.has_edge(v, u): # Only goes from u to v, but not v to u
        one_way_edges.append((u, v))

# Create a graph for one way only to plot
G_one_way = nx.DiGraph()
G_one_way.add_nodes_from(G.nodes(data=True))
G_one_way.add_edges_from(one_way_edges)

print(f"One-way edges: {len(one_way_edges)}")
max_edges_one_direction = 500
plot_map(G_one_way, "Airports Connections (One-way only)", max_edges=max_edges_one_direction, edge_alpha=0.4, directions=True, direction_start_color='green', direction_end_color='red')

One-way edges: 3935
Total edges: 3935
Sampling 500 edges out of 3935 for faster plotting.


## Plot Map with One-Way Connections (In-Degree vs Out-Degree)
Nodes are colored based on their `in_degree - out_degree` in the one-way graph. A red node has more in-degree (more arrivals), while a green node has more out-degree (more departures).

In [ ]:
# Calculate in_degree - out_degree for each node in the one-way graph
degree_diffs = []
for n in G_one_way.nodes():
    in_deg = G_one_way.in_degree(n)
    out_deg = G_one_way.out_degree(n)
    degree_diffs.append(in_deg - out_deg)

# Re-declare plot_map to use RdYlGn_r colorscale for node_color mapping if passed an array
def plot_map_custom_color(graph, title, node_color='blue', edge_color='gray', edge_alpha=0.4, max_edges=3000, directions=False, color_mode='std'):
    fig = go.Figure()
    edges = list(graph.edges())
    print(f"Total edges: {len(edges)}")
    if len(edges) > max_edges:
        edges = random.sample(edges, max_edges)
        print(f"Sampling {max_edges} edges out of {len(graph.edges())} for faster plotting.")
        
    edge_lons, edge_lats = [], []
    for u, v in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        edge_lons.extend([x1, x2, None])
        edge_lats.extend([y1, y2, None])
        
    fig.add_trace(go.Scattergeo(
        lon=edge_lons, lat=edge_lats,
        mode='lines', line=dict(width=1, color=edge_color),
        opacity=edge_alpha, hoverinfo='none', showlegend=False
    ))
    
    node_lons = [pos[n][0] for n in graph.nodes()]
    node_lats = [pos[n][1] for n in graph.nodes()]
    
    hover_texts = []
    for n in graph.nodes():
        attr = graph.nodes[n]
        name = attr.get('name', str(n))
        if graph.is_directed():
            deg_str = f"In-Degree: {graph.in_degree(n)}<br>Out-Degree: {graph.out_degree(n)}"
        else:
            deg_str = f"Degree: {graph.degree(n)}"
        hover_texts.append(f"Airport: {name}<br>{deg_str}")
        
    marker_dict = dict(size=5, opacity=0.8)
    if isinstance(node_color, (list, np.ndarray, tuple)): 
        marker_dict['color'] = node_color
        marker_dict['colorscale'] = 'RdYlGn_r'  # Reversed: low/neg (green -> out > in) to high/pos (red -> in > out)
        
        if color_mode == 'std':
            std_val = np.std(node_color)
            marker_dict['cmin'] = -std_val
            marker_dict['cmax'] = std_val
            
        marker_dict['showscale'] = True
        marker_dict['colorbar'] = dict(thickness=15)
    else:
        marker_dict['color'] = node_color

    fig.add_trace(go.Scattergeo(
        lon=node_lons, lat=node_lats,
        mode='markers', text=hover_texts, hoverinfo='text',
        marker=marker_dict, showlegend=False
    ))
    
    x_coords = [p[0] for p in pos.values()]
    y_coords = [p[1] for p in pos.values()]
    fig.update_layout(
        title=title,
        geo=dict(
            projection_type='mercator',
            showland=True, landcolor='#e0f3d9', showocean=True, oceancolor='#aadaff',
            showcountries=True, countrycolor='#999999',
            lataxis_range=[min(y_coords)-2, max(y_coords)+2], lonaxis_range=[min(x_coords)-2, max(x_coords)+2]
        ),
        margin=dict(l=0, r=0, t=40, b=0), height=600
    )
    fig.show()

max_edges_one_direction = 500
plot_map_custom_color(
    G_one_way, 
    "Airports Connections (One-way only) - Green (More Out) to Red (More In)", 
    node_color=degree_diffs, 
    edge_color='gray', 
    edge_alpha=0.1, 
    max_edges=max_edges_one_direction, 
    directions=False,
    color_mode='std'
)

Total edges: 3935
Sampling 500 edges out of 3935 for faster plotting.


## Bidirectional (Two-way) Connections: Undirected Graph
Extract only connections that exist in both directions, build a new undirected graph, and plot it.

In [ ]:
two_way_edges = []
for u, v in G.edges():
    if G.has_edge(v, u):
        # Check u < v to avoid duplicate edge entries in our undirected setup
        if u < v:
            two_way_edges.append((u, v))

G_undirected = nx.Graph()
G_undirected.add_nodes_from(G.nodes(data=True))
G_undirected.add_edges_from(two_way_edges)

print(f"Two-way edges extracted: {len(two_way_edges)}")

plot_map(G_undirected, 
         f"Bidirectional Airport Connections (Undirected Graph)\nNodes: {G_undirected.number_of_nodes()}, Edges: {G_undirected.number_of_edges()}", 
         edge_alpha=0.4, max_edges=200)

Two-way edges extracted: 34012
Total edges: 34012
Sampling 200 edges out of 34012 for faster plotting.


## Analysis of the undirected graph.

In [ ]:
N_undir = G_undirected.number_of_nodes()
L_undir = G_undirected.number_of_edges()

avg_degree_undir = sum(d for n, d in G_undirected.degree()) / N_undir
density_undir = nx.density(G_undirected)

connected_components = list(nx.connected_components(G_undirected))
num_cc = len(connected_components)
largest_cc_size = max(len(c) for c in connected_components)

print("--- Undirected Graph ---")
print(f"Nodes (N): {N_undir}")
print(f"Edges (L): {L_undir}")
print(f"Average degree (2L/N): {avg_degree_undir:.4f}")
print(f"Density (2L / N(N-1)): {density_undir:.6f}")
print(f"Connected components: {num_cc}")

--- Undirected Graph ---
Nodes (N): 456
Edges (L): 34012
Average degree (2L/N): 149.1754
Density (2L / N(N-1)): 0.327858
Connected components: 1


In [ ]:
import random
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# Configuration options as requested
change_size = True
show_edges = True
max_edges = 3000

# Calculate centralities (using Undirected graph G_undirected)
closeness = nx.closeness_centrality(G_undirected)
betweenness = nx.betweenness_centrality(G_undirected)
try:
    eigenvector = nx.eigenvector_centrality_numpy(G_undirected)
except:
    eigenvector = nx.eigenvector_centrality(G_undirected, max_iter=1000)

try:
    katz = nx.katz_centrality_numpy(G_undirected)
except:
    katz = nx.katz_centrality(G_undirected, max_iter=1000)

centrality_measures = {
    "Closeness Centrality": closeness,
    "Betweenness Centrality": betweenness,
    "Eigenvector Centrality": eigenvector,
    "Katz Centrality": katz
}

fig = make_subplots(
    rows=2, cols=2, 
    specs=[[{"type": "geo"}, {"type": "geo"}], [{"type": "geo"}, {"type": "geo"}]],
    subplot_titles=list(centrality_measures.keys()),
    vertical_spacing=0.08, horizontal_spacing=0.05
)

x_coords = [p[0] for p in pos.values()]
y_coords = [p[1] for p in pos.values()]
lon_range = [min(x_coords)-2, max(x_coords)+2]
lat_range = [min(y_coords)-2, max(y_coords)+2]

# Pre-calculate edges logic if we need to draw them
if show_edges:
    edges = list(G_undirected.edges())
    if len(edges) > max_edges:
        edges = random.sample(edges, max_edges)
    edge_lons, edge_lats = [], []
    for u, v in edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        edge_lons.extend([x1, x2, None])
        edge_lats.extend([y1, y2, None])

nodes = list(G_undirected.nodes())
node_lons = [pos[n][0] for n in nodes]
node_lats = [pos[n][1] for n in nodes]

for i, (title, measure) in enumerate(centrality_measures.items()):
    row = (i // 2) + 1
    col = (i % 2) + 1
    
    if show_edges:
        fig.add_trace(go.Scattergeo(
            lon=edge_lons, lat=edge_lats,
            mode='lines', line=dict(width=1, color='gray'),
            opacity=0.4, hoverinfo='none', showlegend=False
        ), row=row, col=col)
        
    values = [measure[node] for node in nodes]
    val_min = min(values)
    val_max = max(values)
    val_range = val_max - val_min if val_max > val_min else 1
    
    if change_size:
        sizes = [((v - val_min) / val_range) * 20 + 3 for v in values] # Dynamic size for plotly markers
    else:
        sizes = [6 for _ in values]
        
    hover_texts = []
    for n, v in zip(nodes, values):
        attr = G_undirected.nodes[n]
        name = attr.get('name', str(n))
        if G_undirected.is_directed():
            deg_str = f"In-Degree: {G_undirected.in_degree(n)}<br>Out-Degree: {G_undirected.out_degree(n)}"
        else:
            deg_str = f"Degree: {G_undirected.degree(n)}"
        hover_texts.append(f"Airport: {name}<br>{deg_str}<br>Value: {v:.4f}")
        
    # To avoid colorbars overlapping, position them independently
    cb_x = 0.45 if col == 1 else 1.0
    cb_y = 0.78 if row == 1 else 0.22
    
    fig.add_trace(go.Scattergeo(
        lon=node_lons, lat=node_lats,
        mode='markers', text=hover_texts, hoverinfo='text',
        marker=dict(
            size=sizes, color=values, colorscale='Viridis',
            showscale=True,
            colorbar=dict(thickness=10, len=0.42, xanchor='left', x=cb_x, y=cb_y)
        ),
        showlegend=False
    ), row=row, col=col)
    
    fig.update_geos(
        projection_type='mercator',
        showland=True, landcolor='#e0f3d9',
        showocean=True, oceancolor='#aadaff',
        showcountries=True, countrycolor='#999999',
        lataxis_range=lat_range, lonaxis_range=lon_range,
        row=row, col=col
    )

fig.update_layout(height=1000, margin=dict(l=0, r=0, t=50, b=0))
fig.show()


### TODO ESTUDIAR CORRELACIÓN ENTRE LAS MEDIDAS DE CENTRALIDAD. 

In [ ]:
import networkx as nx
import plotly.graph_objects as go

# Extract the largest connected component (LCC) for distance-based calculations
# (Mean shortest path, diameter, and radius require a fully connected graph)
largest_cc = max(nx.connected_components(G_undirected), key=len)
G_lcc = G_undirected.subgraph(largest_cc)

# 1. Average Clustering Coefficient (computed on the entire undirected graph)
avg_clustering_watts_strogatz = nx.average_clustering(G_undirected)
avg_clustering_newman = nx.transitivity(G_undirected)

# 2. Distance metrics (computed on the LCC)
mean_shortest_path = nx.average_shortest_path_length(G_lcc)


diameter = nx.diameter(G_lcc)
radius = nx.radius(G_lcc)

print("--- Undirected Graph Metrics ---")
print(f"Watts-Strogatz Clustering Coefficient (Average): {avg_clustering_watts_strogatz:.4f}")
print(f"Newman Clustering Coefficient (Transitivity): {avg_clustering_newman:.4f}")
print(f"\n--- Distance Metrics (Largest Connected Component: {len(G_lcc)} nodes) ---")
print(f"Mean Shortest Path: {mean_shortest_path:.4f}")
print(f"Diameter: {diameter}")
print(f"Radius: {radius}")

# 3. Node Degree Histogram
degrees = [d for n, d in G_undirected.degree()]

fig_hist = go.Figure(data=[go.Histogram(
    x=degrees,
    nbinsx=50,
    marker_color='#1f77b4',
    opacity=0.8
)])

fig_hist.update_layout(
    title="Node Degree Histogram (Undirected Graph)",
    xaxis_title="Degree",
    yaxis_title="Frequency / Number of Nodes",
    template="plotly_white",
    bargap=0.1
)

fig_hist.show()

--- Undirected Graph Metrics ---
Watts-Strogatz Clustering Coefficient (Average): 0.8068
Newman Clustering Coefficient (Transitivity): 0.5908

--- Distance Metrics (Largest Connected Component: 456 nodes) ---
Mean Shortest Path: 1.6747
Diameter: 3
Radius: 2


In [ ]:
import random
import plotly.graph_objects as go
import numpy as np

# Calculate local clustering coefficient for each node
clustering_coeffs = nx.clustering(G_undirected)
nodes = list(G_undirected.nodes())
cc_values = [clustering_coeffs[n] for n in nodes]

# Calculate dynamic sizes
val_min = min(cc_values)
val_max = max(cc_values)
val_range = val_max - val_min if val_max > val_min else 1
sizes = [((v - val_min) / val_range) * 20 + 3 for v in cc_values]

fig = go.Figure()

# Edges
max_edges = 3000
edges = list(G_undirected.edges())
if len(edges) > max_edges:
    edges = random.sample(edges, max_edges)

edge_lons, edge_lats = [], []
for u, v in edges:
    if u in pos and v in pos:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        edge_lons.extend([x1, x2, None])
        edge_lats.extend([y1, y2, None])

fig.add_trace(go.Scattergeo(
    lon=edge_lons, lat=edge_lats,
    mode='lines', line=dict(width=1, color='gray'),
    opacity=0.4, hoverinfo='none', showlegend=False
))

# Nodes
node_lons = [pos[n][0] for n in nodes if n in pos]
node_lats = [pos[n][1] for n in nodes if n in pos]

hover_texts = []
for n, v in zip(nodes, cc_values):
    attr = G_undirected.nodes[n]
    name = attr.get('name', str(n))
    deg = G_undirected.degree(n)
    hover_texts.append(f"Airport: {name}<br>Degree: {deg}<br>Clustering Coeff: {v:.4f}")

fig.add_trace(go.Scattergeo(
    lon=node_lons, lat=node_lats,
    mode='markers', text=hover_texts, hoverinfo='text',
    marker=dict(
        size=sizes,
        color=cc_values,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(thickness=15, title="Clust. Coeff")
    ),
    showlegend=False
))

# Calculate native boundaries
x_coords = [p[0] for p in pos.values()]
y_coords = [p[1] for p in pos.values()]

fig.update_layout(
    title="Local Clustering Coefficient Map (Undirected Graph)",
    geo=dict(
        projection_type='mercator',
        showland=True, landcolor='#e0f3d9', showocean=True, oceancolor='#aadaff',
        showcountries=True, countrycolor='#999999',
        lataxis_range=[min(y_coords)-2, max(y_coords)+2], lonaxis_range=[min(x_coords)-2, max(x_coords)+2]
    ),
    margin=dict(l=0, r=0, t=40, b=0), height=600
)

fig.show()

### TODO CALCULAR CLUSTERING COEFFICIENT MEDIO

In [ ]:
import random
import networkx as nx
import plotly.graph_objects as go

# Ensure we're using the Largest Connected Component since center/periphery require measurable distances
center_nodes = set(nx.center(G_lcc))
periphery_nodes = set(nx.periphery(G_lcc))

print(f"Center nodes: {len(center_nodes)}")
print(f"Periphery nodes: {len(periphery_nodes)}")

fig = go.Figure()

# Plot Edges
max_edges = 3000
edges = list(G_lcc.edges())
if len(edges) > max_edges:
    edges = random.sample(edges, max_edges)

edge_lons, edge_lats = [], []
for u, v in edges:
    if u in pos and v in pos:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        edge_lons.extend([x1, x2, None])
        edge_lats.extend([y1, y2, None])

fig.add_trace(go.Scattergeo(
    lon=edge_lons, lat=edge_lats,
    mode='lines', line=dict(width=1, color='gray'),
    opacity=0.4, hoverinfo='none', showlegend=False
))

# Categorize Nodes
node_lons_c, node_lats_c, text_c = [], [], []
node_lons_p, node_lats_p, text_p = [], [], []
node_lons_o, node_lats_o, text_o = [], [], []

for n in G_lcc.nodes():
    if n in pos:
        attr = G_lcc.nodes[n]
        name = attr.get('name', str(n))
        deg = G_lcc.degree(n)
        
        info = f"Airport: {name}<br>Degree: {deg}"
        
        if n in center_nodes:
            node_lons_c.append(pos[n][0])
            node_lats_c.append(pos[n][1])
            text_c.append(f"{info}<br>Type: <b>Center</b>")
        elif n in periphery_nodes:
            node_lons_p.append(pos[n][0])
            node_lats_p.append(pos[n][1])
            text_p.append(f"{info}<br>Type: <b>Periphery</b>")
        else:
            node_lons_o.append(pos[n][0])
            node_lats_o.append(pos[n][1])
            text_o.append(f"{info}<br>Type: Other")

# Trace for 'Other' nodes
fig.add_trace(go.Scattergeo(
    lon=node_lons_o, lat=node_lats_o,
    mode='markers', text=text_o, hoverinfo='text',
    marker=dict(size=5, color='black', opacity=1),
    name='Other'
))

# Trace for 'Periphery' nodes
fig.add_trace(go.Scattergeo(
    lon=node_lons_p, lat=node_lats_p,
    mode='markers', text=text_p, hoverinfo='text',
    marker=dict(size=8, color='blue', opacity=1),
    name='Periphery'
))

# Trace for 'Center' nodes
fig.add_trace(go.Scattergeo(
    lon=node_lons_c, lat=node_lats_c,
    mode='markers', text=text_c, hoverinfo='text',
    marker=dict(size=10, color='red', opacity=1.0, line=dict(width=1, color='darkred')),
    name='Center'
))

# Layout configurations
x_coords = [p[0] for p in pos.values()]
y_coords = [p[1] for p in pos.values()]

fig.update_layout(
    title="Center (Red) and Periphery (Blue) Nodes",
    geo=dict(
        projection_type='mercator',
        showland=True, landcolor='#e0f3d9', showocean=True, oceancolor='#aadaff',
        showcountries=True, countrycolor='#999999',
        lataxis_range=[min(y_coords)-2, max(y_coords)+2], lonaxis_range=[min(x_coords)-2, max(x_coords)+2]
    ),
    margin=dict(l=0, r=0, t=40, b=0), height=600,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

Center nodes: 348
Periphery nodes: 108


All nodes are either center nodes or periphery nodes

In [ ]:
# Degree PMF on log-log with fixed axis ranges (k: 10^1–10^3, P(k): 10^-1–10^-5)
import collections
import numpy as np
import plotly.graph_objects as go

# Toggle: show/hide the power-law fit line
show_powerlaw_fit = False

# Degrees (undirected)
degrees = [d for n, d in G_undirected.degree()]
if len(degrees) == 0:
    print("No degrees found in G_undirected")
else:
    counter = collections.Counter(degrees)
    degs, freqs = zip(*sorted(counter.items()))
    degs = np.array(degs)
    freqs = np.array(freqs)

    # Use PMF P(k) = freq / sum(freq)
    pmf = freqs / freqs.sum()

    # Ignore zeros for log transform
    mask = (degs > 0) & (pmf > 0)

    # Fit a line in log10 space (on the PMF)
    slope = None
    intercept = None
    if mask.sum() >= 2:
        a, b = np.polyfit(np.log10(degs[mask]), np.log10(pmf[mask]), 1)
        slope = a
        intercept = b
        x_fit = np.logspace(1, 3, 200)  # from 10^1 to 10^3
        y_fit = (10 ** intercept) * (x_fit ** slope)
    else:
        x_fit = np.array([])
        y_fit = np.array([])

    fig = go.Figure()
    # PMF points
    fig.add_trace(go.Scatter(
        x=degs[mask], y=pmf[mask],
        mode='markers', marker=dict(size=6, color='black', opacity=0.8),
        name='P(k)'
    ))

    # Optional fit (controlled by flag)
    if show_powerlaw_fit and slope is not None:
        fig.add_trace(go.Scatter(
            x=x_fit, y=y_fit,
            mode='lines', line=dict(color='red', dash='dash'),
            name=f'Power-law fit (slope={slope:.2f})'
        ))

    # Set log axes and ranges and show only decade ticks (Unicode superscripts)
    fig.update_xaxes(type='log', range=[1, 3], title_text='k',
                     tickmode='array',
                     tickvals=[10, 100, 1000],
                     ticktext=['10', '10²', '10³'],
                     showline=True, linecolor='black', linewidth=1,
                     ticks='outside', ticklen=6, tickwidth=1, showgrid=False, automargin=True)

    # Restore y-axis orientation to 10^-5 (bottom) -> 10^0 (top)
    fig.update_yaxes(type='log', range=[-5, 0], title_text='P(k)',
                     tickmode='array',
                     tickvals=[1e-1, 1e-2, 1e-3, 1e-4, 1e-5],
                     ticktext=['10⁻¹', '10⁻²', '10⁻³', '10⁻⁴', '10⁻⁵'],
                     showline=True, linecolor='black', linewidth=1,
                     ticks='outside', ticklen=6, tickwidth=1, showgrid=False, automargin=True)

    fig.update_layout(
        title='Degree distribution PMF (log-log) - Undirected graph',
        template='plotly_white',
        width=900, height=600,
        margin=dict(l=90, r=20, t=70, b=90)
    )

    fig.show()